# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SupreetOP/Flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue ranks content items according to the measured opportunity signals observed in the available Search Console and Analytics features. The goal is not to automate content decisions, but to help a content team review the highest-priority opportunities first.

### Ranked actions

| Action | Description |
|----------|----------|
| REFRESH | Content shows measurable search visibility but appears to underperform relative to its opportunity. A content review is recommended. |
| MONITOR | Content does not currently show a strong refresh signal. Continue monitoring and reassess during future review cycles. |

### Reason codes

| Reason Code | Meaning |
|-------------|----------|
| HIGH_IMPRESSIONS_LOW_CTR | The content receives meaningful search impressions but a relatively low click-through rate. This may indicate an opportunity to improve titles, descriptions, content relevance, or search intent alignment. |
| LOW_SEARCH_SIGNAL | The content currently shows limited search visibility. Additional evidence is needed before recommending a refresh action. |

### Priority order

The ranked queue should be reviewed from highest score to lowest score. Higher-ranked items represent stronger measured signals according to the model and baseline rules. The queue is intended as decision-support and should not be treated as automatic approval for content changes.

In [14]:

# RECREATE WEEK-4 BASELINE ACTION QUEUE


!pip -q install duckdb

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os


#  Connect to Hugging Face


HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
""")


#  Build March 2026 feature vector


queue = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Feature rows:", len(queue))


# Calculate CTR


queue["ctr"] = np.where(
    queue["gsc_impressions"] > 0,
    queue["gsc_clicks"] / queue["gsc_impressions"],
    0
)


#  Week-4 baseline thresholds


MIN_IMPRESSIONS = 100
MAX_CTR = 0.02

high_impressions = (
    queue["gsc_impressions"] >= MIN_IMPRESSIONS
)

low_ctr = (
    queue["ctr"] < MAX_CTR
)


#  Define refresh opportunity


refresh_opportunity = (
    high_impressions &
    low_ctr
)


# Calculate baseline score

queue["score"] = np.where(
    refresh_opportunity,
    np.log1p(queue["gsc_impressions"])
    * (1 - queue["ctr"].clip(0, 1)),
    0
)


#  Reason code


queue["reason_code"] = np.where(
    refresh_opportunity,
    "HIGH_IMPRESSIONS_LOW_CTR",
    "LOW_SEARCH_SIGNAL"
)


#  Action


queue["action"] = np.where(
    refresh_opportunity,
    "REFRESH",
    "MONITOR"
)


#  Rank


queue = queue.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)


#  Final ranked queue


baseline_queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].copy()


#  Export


os.makedirs(
    "work/outputs",
    exist_ok=True
)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)


#  Verification


print("=" * 70)
print("BASELINE QUEUE CREATED")
print("=" * 70)

print(f"Saved: {output_path}")
print(f"Rows: {len(baseline_queue):,}")

print()
print("RULE THRESHOLDS")
print("-" * 70)
print(f"Minimum impressions: {MIN_IMPRESSIONS}")
print(f"Maximum CTR: {MAX_CTR:.2%}")

print()
print("ACTION COUNTS")
print("-" * 70)
print(baseline_queue["action"].value_counts())

print()
print("REASON CODE COUNTS")
print("-" * 70)
print(baseline_queue["reason_code"].value_counts())

print()
print("TOP 10 RANKED QUEUE")
print("-" * 70)

display(baseline_queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 331437
BASELINE QUEUE CREATED
Saved: work/outputs/baseline_action_score.csv
Rows: 331,437

RULE THRESHOLDS
----------------------------------------------------------------------
Minimum impressions: 100
Maximum CTR: 2.00%

ACTION COUNTS
----------------------------------------------------------------------
action
MONITOR    230777
REFRESH    100660
Name: count, dtype: int64

REASON CODE COUNTS
----------------------------------------------------------------------
reason_code
LOW_SEARCH_SIGNAL           230777
HIGH_IMPRESSIONS_LOW_CTR    100660
Name: count, dtype: int64

TOP 10 RANKED QUEUE
----------------------------------------------------------------------


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,13.210371,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,12.374843,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,12.335260,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
3,4,client_e547b89c05043229,content_0e03de7680314cd5,12.267284,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
4,5,client_23a62021009f63c4,content_44f34c0a90047651,12.264864,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
5,6,client_e547b89c05043229,content_8d7d99f109e19aa2,12.206052,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
6,7,client_62f4a7e64f5e0096,content_7172a7fad43f0998,12.183760,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,12.163452,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,12.154734,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,12.123935,HIGH_IMPRESSIONS_LOW_CTR,REFRESH


## 2. Intended use and limits

### Intended use

The action queue is intended for content teams as a decision-support tool for prioritizing which content items should receive human review first.

The ranked score uses observed search visibility and engagement signals available at the decision moment. Higher-ranked items with the `HIGH_IMPRESSIONS_LOW_CTR` reason code are candidates for a content refresh review, while `LOW_SEARCH_SIGNAL` items are better treated as monitor cases.

The queue is intended to help allocate limited review time. It does not determine that a page must be refreshed.

### Limits

The recommendations are directional rather than definitive. The available features do not capture every factor that can affect content performance, including search intent, SERP features, content quality, seasonality, competition, technical issues, or changes in user demand.

A high score therefore means that the observed signals match the baseline's refresh rule; it does not prove that changing the content will increase future traffic or clicks.

The queue should be used for prioritization and human review, not as an automated publishing, deletion, or content-change system.

In [15]:

#  QUEUE COVERAGE CHECK


print("=" * 70)
print("INTENDED USE — QUEUE SUMMARY")
print("=" * 70)

print(f"Total content items: {len(baseline_queue):,}")

print("\nAction distribution:")
display(
    baseline_queue["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

print("\nReason-code distribution:")
display(
    baseline_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)


INTENDED USE — QUEUE SUMMARY
Total content items: 331,437

Action distribution:


,action,count
0,MONITOR,230777
1,REFRESH,100660



Reason-code distribution:


,reason_code,count
0,LOW_SEARCH_SIGNAL,230777
1,HIGH_IMPRESSIONS_LOW_CTR,100660


## 3. Human review + the no-go list

### Human review requirements

A content reviewer should check a recommended item before taking action.

Before refreshing content, the reviewer should verify:

1. **Search intent** — Does the page actually match what users are searching for?
2. **SERP context** — Are search features, competitors, or other results affecting the click-through rate?
3. **Content quality** — Is the content outdated, incomplete, inaccurate, or otherwise in need of improvement?
4. **Technical factors** — Could indexing, page speed, canonicalization, or other technical issues explain the observed performance?
5. **Business relevance** — Is the page still relevant and valuable to the intended audience?
6. **Recent changes** — Has the page, query landscape, or demand changed recently in a way that makes the historical signals less reliable?

A reviewer should be able to reject or downgrade a recommendation when the contextual evidence does not support the suggested action.

### No-go list

The system should NOT automatically:

- publish or modify content;
- delete or redirect pages;
- change titles or metadata without human review;
- declare a page successful or unsuccessful based only on the model score;
- make decisions about business-critical or sensitive content without review;
- treat a high score as proof that a refresh will increase traffic or clicks;
- override expert judgment when the available data does not explain the recommendation.

The queue is therefore a prioritization tool, not an autonomous content-management system.

In [16]:

#  HUMAN REVIEW / NO-GO CHECK


print("=" * 70)
print("HUMAN REVIEW CHECK")
print("=" * 70)

print("Total ranked recommendations:", len(baseline_queue))

print("\nActions requiring human review:")
print(baseline_queue["action"].value_counts())

print("\nAutomated actions allowed:")
print("NONE")

print("\nNO-GO:")
print("No publishing, deletion, redirection, metadata changes,")
print("or other content changes should be automated from this queue.")

HUMAN REVIEW CHECK
Total ranked recommendations: 331437

Actions requiring human review:
action
MONITOR    230777
REFRESH    100660
Name: count, dtype: int64

Automated actions allowed:
NONE

NO-GO:
No publishing, deletion, redirection, metadata changes,
or other content changes should be automated from this queue.


## 4. Monitoring / retrain triggers

The recommendations should be considered stale if the relationship between the observed decision-time signals and later content performance changes materially.

### Monitoring signals

The following should be checked during each review cycle:

- The distribution of `gsc_impressions` and `gsc_clicks` compared with the data used to create the queue.
- The proportion of content receiving `REFRESH` versus `MONITOR`.
- The distribution of recommendation scores.
- The observed future click rate for previously recommended content.
- The proportion of recommended items that human reviewers reject or downgrade.

### Suggested review triggers

A new model or baseline review should be considered when:

1. The feature distributions shift substantially from the training/decision-time data.
2. The percentage of `REFRESH` recommendations changes materially without a known business reason.
3. The measured performance of recommended content deteriorates over repeated evaluation periods.
4. Human reviewers frequently reject recommendations for the same reason.
5. Important input fields become unavailable, change definition, or develop unusual missingness.

These are monitoring and review triggers, not automatic retraining rules. A human should investigate the cause before changing the model or baseline.

In [17]:

#  CURRENT QUEUE MONITORING SNAPSHOT


print("=" * 70)
print("MONITORING / RETRAINING SNAPSHOT")
print("=" * 70)

print("\n1. Action distribution")
print("-" * 70)
action_counts = baseline_queue["action"].value_counts()
display(action_counts)

print("\n2. Action percentages")
print("-" * 70)
action_percent = (
    baseline_queue["action"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
display(action_percent)

print("\n3. Score distribution")
print("-" * 70)
display(
    baseline_queue["score"].describe()
)

print("\n4. Reason-code distribution")
print("-" * 70)
display(
    baseline_queue["reason_code"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMonitoring baseline created.")
print("Future review cycles can compare these distributions against")
print("new recommendation runs to identify potential drift.")


MONITORING / RETRAINING SNAPSHOT

1. Action distribution
----------------------------------------------------------------------


,count
action,
MONITOR,230777
REFRESH,100660



2. Action percentages
----------------------------------------------------------------------


,proportion
action,
MONITOR,69.63
REFRESH,30.37



3. Score distribution
----------------------------------------------------------------------


,score
count,331437.000000
mean,2.067332
std,3.224712
min,0.000000
25%,0.000000
50%,0.000000
75%,5.347108
max,13.210371



4. Reason-code distribution
----------------------------------------------------------------------


,proportion
reason_code,
LOW_SEARCH_SIGNAL,69.63
HIGH_IMPRESSIONS_LOW_CTR,30.37



Monitoring baseline created.
Future review cycles can compare these distributions against
new recommendation runs to identify potential drift.


## 5. Exports for the paper

The ranked action queue is exported as a CSV so that the paper can reuse the same decision-support output without manually copying results from the notebook.

The exported queue contains the rank, hashed client and content identifiers, score, reason code, and recommended action.

The queue is generated by the notebook and is not treated as a production data artifact. The paper should describe the recommendations using careful, directional language and should retain the human-review limitations documented above.

In [18]:

#  EXPORTS FOR THE PAPER


import os
import pandas as pd

# Make sure the output directory exists
os.makedirs("work/outputs", exist_ok=True)

# Export the ranked queue
queue_output = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    queue_output,
    index=False
)

print("=" * 70)
print("PAPER EXPORT")
print("=" * 70)

print(f"File: {queue_output}")
print(f"Rows: {len(baseline_queue):,}")
print(f"Columns: {list(baseline_queue.columns)}")

print("\nFile exists:", os.path.exists(queue_output))

print("\nExport preview:")
display(pd.read_csv(queue_output).head(10))


PAPER EXPORT
File: work/outputs/baseline_action_score.csv
Rows: 331,437
Columns: ['rank', 'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action']

File exists: True

Export preview:


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,13.210371,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,12.374843,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,12.335260,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
3,4,client_e547b89c05043229,content_0e03de7680314cd5,12.267284,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
4,5,client_23a62021009f63c4,content_44f34c0a90047651,12.264864,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
5,6,client_e547b89c05043229,content_8d7d99f109e19aa2,12.206052,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
6,7,client_62f4a7e64f5e0096,content_7172a7fad43f0998,12.183760,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,12.163452,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,12.154734,HIGH_IMPRESSIONS_LOW_CTR,REFRESH
9,10,client_62f4a7e64f5e0096,content_f107e54b10b43725,12.123935,HIGH_IMPRESSIONS_LOW_CTR,REFRESH


## Self-check

- [x] Every section above is filled with the required markdown explanations and supporting code/results.
- [x] The notebook creates a ranked action queue with reason codes.
- [x] The intended use and limitations of the recommendations are documented.
- [x] Human review requirements and explicit no-go cases are documented.
- [x] Monitoring and retrain/review triggers are defined using measurable signals.
- [x] The ranked queue is exported to `work/outputs/baseline_action_score.csv`.
- [x] Claims are stated carefully using words such as observed, measured, directional, and decision-support.
- [x] The notebook runs from top to bottom without errors using Runtime → Run all.
- [x] No client names, URLs, or private queries are included in the notebook.
- [x] The notebook has been committed and pushed to `work/notebooks/w07_action_playbook.ipynb`.
- [x] The repository URL has been submitted on the assignment card.